# DL_03: BioCause Directed Relation Classification with PubMedBERT
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook implements fine-tuning of **PubMedBERT Fulltext** (`microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`) for candidate-conditioned directed causal relation classification on the **BioCause** corpus.

Key requirements:
- Predict directed causal relations ($1 = \text{ARG1 causes ARG2}$, $0 = \text{ARG1 does not cause ARG2 under candidate convention}$).
- Retain explicit argument boundary markers: `[E1]...[/E1]` for Arg1 and `[E2]...[/E2]` for Arg2 in contextual text.
- Standardize on checkpoint `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`.
- Handle class imbalance using dynamically computed training loss weights ($w_{\text{pos}} = 9,307 / 621 \approx 14.9871$). Never hardcode or derive weights from validation/test sets.
- Use conservative GPU defaults: Batch size 8, gradient accumulation 1, FP16 when supported on CUDA, `eval_strategy='epoch'`.
- Generate predictions for **both** validation and test splits:
  - `data/processed/dl_biocause_validation_predictions.csv` (1,884 rows)
  - `data/processed/dl_biocause_test_predictions.csv` (1,394 rows)
- Include required columns: `candidate_id, article_id, section_id, arg1_id, arg2_id, true_label, dl_prob, dl_pred, split`.
- Preserve raw predicted probabilities; use validation split strictly for decision threshold tuning; never use test labels for tuning.


## 2. Experimental Methodology & Theoretical Background

### Biomedical Domain Pre-training (PubMedBERT Fulltext)
- **PubMedBERT Fulltext** is pretrained *from scratch* on 14M PubMed abstracts and full-text PMC articles with a domain-specific vocabulary.
- It consistently outperforms general-domain BERT on biomedical relation extraction benchmarks.

### Directional Candidate Semantics & Class Imbalance
- **Candidate Generation**: All co-occurring entity pairs within sentences are extracted as directed candidates $(Arg_1, Arg_2)$.
- **Label Semantics**: Label `1` represents directed causation $Arg_1 \to Arg_2$. Label `0` indicates that $Arg_1$ does not cause $Arg_2$ in that candidate configuration. Label `0` includes reversed directional candidates $(Arg_2, Arg_1)$ and non-causal pairs.
- **Dynamic Imbalance Mitigation**: Model-eligible dataset preparation produces 9,928 training pairs (621 positive, 9,307 negative), giving an actual training ratio of **14.99:1**.
- The loss function incorporates a dynamic positive class weight: $w_{\text{pos}} = \frac{N_{\text{train, neg}}}{N_{\text{train, pos}}} = \frac{9307}{621} \approx 14.9871$.


## 3. Expected Inputs

| Split | Source File | Articles | Gold Relations | Model-Eligible Candidates | Neg / Pos Ratio |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Train** | `data/processed/dl_biocause_train.csv` | 12 PMC Articles | 621 | 9,928 | 14.99 : 1 |
| **Validation** | `data/processed/dl_biocause_val.csv` | 4 PMC Articles | 131 | 1,884 | 13.38 : 1 |
| **Test** | `data/processed/dl_biocause_test.csv` | 4 PMC Articles | 100 | 1,394 | 12.94 : 1 |


## 4. Technical Responsibilities & Implementation Requirements

1. **Portable Root Setup**: Resolve repository path dynamically across environments.
2. **Checkpoint & Tokenizer Configuration**: Standardize on `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` and register special tokens `['[E1]', '[/E1]', '[E2]', '[/E2]']`.
3. **Embedding Resizing**: Expand model token embeddings to accommodate newly added boundary tokens.
4. **Dynamic Weighted Loss**: Calculate $w_{\text{pos}}$ dynamically from training counts and configure weighted cross-entropy loss.
5. **Conservative Training Configuration**: Batch size 8, gradient accumulation 1, FP16 enabled on CUDA, `eval_strategy='epoch'`.
6. **Validation Prediction & Decision Threshold Tuning**: Predict probabilities on the 4 validation articles (1,884 candidates) and tune decision threshold $\tau^*$ to optimize validation F1.
7. **Held-Out Test Set Prediction**: Predict probabilities on the 1,394 test candidates using frozen $\tau^*$.
8. **Export Dual Prediction Artifacts**: Save both validation and test predictions to `data/processed/` for hybrid integration in `DL_04`.


In [1]:
# Section 1: Environment Setup & Portable Imports
import sys
import os
import json
import warnings
import re
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Portability setup across Local repo, Colab, and Kaggle
def find_repo_root():
    curr = Path(os.getcwd()).resolve()
    preferred_roots = [Path('/content/causal-nlp-reasoning'), Path('/workspace/causal-nlp-reasoning'), Path('/kaggle/working/causal-nlp-reasoning')]
    search_roots = [*preferred_roots, curr, Path('/content'), Path('/workspace'), Path('/kaggle/working')]
    for base in search_roots:
        if not base.exists():
            continue
        candidates = [base, *base.parents]
        if base in {Path('/content'), Path('/workspace'), Path('/kaggle/working')}:
            candidates.extend(path.parent.parent for path in base.rglob('DL/dl_utils.py'))
        for candidate in candidates:
            if ((candidate / 'AML' / 'causal_extractors.py').is_file()
                    and (candidate / 'DL' / 'dl_utils.py').is_file()):
                return candidate.resolve()
    raise FileNotFoundError(
        'Repository root not found. Make the project folder available in the notebook workspace.'
    )
ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

import dl_utils

# Enforce global determinism
SEED = 42
dl_utils.set_seed(SEED)

In [2]:
# Section 2: Configurable Hyperparameters, Hardware Detection & Dataset Ingestion
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding
import numpy as np
import pandas as pd

PUBMEDBERT_CHECKPOINT = dl_utils.PUBMEDBERT_CHECKPOINT
assert PUBMEDBERT_CHECKPOINT == 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
device = dl_utils.get_device()
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
FP16_ENABLED = bool(cuda_available and not mps_available)

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
MAX_LENGTH = 384
EVAL_STRATEGY = 'epoch'
SAVE_TOTAL_LIMIT = NUM_EPOCHS
OUTPUT_DIR = str(ROOT_DIR / 'DL' / 'checkpoints' / 'biocause_pubmedbert')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DATA_DIR = ROOT_DIR / 'data' / 'processed'

def resolve_split_path(filename):
    candidates = [DATA_DIR / filename, ROOT_DIR / 'data' / 'processed' / filename]
    if Path('/content').exists():
        candidates.extend(Path('/content').rglob(filename))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return DATA_DIR / filename

PRETRAIN_SPLIT_FILES = {
    'train': resolve_split_path('dl_biocause_train.csv'),
    'validation': resolve_split_path('dl_biocause_val.csv'),
}
REQUIRED_COLUMNS = [
    'candidate_id', 'article_id', 'section_id', 'arg1_id', 'arg2_id',
    'marked_text', 'label', 'split'
]
TARGET_COUNTS = dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS

AUTHORITATIVE_VAL_CANDIDATE_IDS = pd.read_csv(
    PRETRAIN_SPLIT_FILES['validation'], usecols=['candidate_id']
)['candidate_id'].tolist()

dfs = {}
for split_name, split_path in PRETRAIN_SPLIT_FILES.items():
    if not split_path.exists():
        raise FileNotFoundError(f'Missing prepared BioCause {split_name} split.')
    df = pd.read_csv(split_path)
    dl_utils.validate_dataset_schema(df, REQUIRED_COLUMNS, dataset_name=f'BioCause-{split_name}')
    assert 'arg1_before_arg2' in df.columns, f'{split_name}: missing arg1_before_arg2 column'
    assert set(df['label'].unique()) == {0, 1}, f'{split_name}: both labels {{0, 1}} must occur'
    assert not df['marked_text'].isna().any() and not df['label'].isna().any()
    for marker in dl_utils.ENTITY_MARKERS:
        assert df['marked_text'].str.contains(re.escape(marker)).all(), (
            f'{split_name}: missing marker {marker}'
        )
    assert all(
        row.find('[E1]') < row.find('[/E1]') and row[row.find('[E1]') + 4:row.find('[/E1]')].strip() != ''
        and row.find('[E2]') < row.find('[/E2]') and row[row.find('[E2]') + 4:row.find('[/E2]')].strip() != ''
        for row in df['marked_text']
    ), f'{split_name}: empty or malformed entity span'
    expected = TARGET_COUNTS[split_name]
    assert len(df) == expected['total']
    assert int((df['label'] == 1).sum()) == expected['positive']
    assert int((df['label'] == 0).sum()) == expected['negative']
    assert df['candidate_id'].is_unique
    dfs[split_name] = df

train_df = dfs['train']
val_df = dfs['validation']
assert set(train_df['article_id']).isdisjoint(set(val_df['article_id']))
n_pos_train = int((train_df['label'] == 1).sum())
n_neg_train = int((train_df['label'] == 0).sum())
pos_weight = dl_utils.calculate_dynamic_class_weight(n_negative=n_neg_train, n_positive=n_pos_train)
assert np.isclose(pos_weight, 9307 / 621)
assert n_pos_train > 0 and n_neg_train > 0

split_summary_df = pd.DataFrame([
    {
        'Split': split_name,
        'Articles': int(df['article_id'].nunique()),
        'Candidates': int(len(df)),
        'Positive_1': int((df['label'] == 1).sum()),
        'Negative_0': int((df['label'] == 0).sum()),
        'Pos_Class_Weight': round(pos_weight, 4) if split_name == 'train' else '-',
    }
    for split_name, df in [('train', train_df), ('validation', val_df)]
])
split_summary_df

,Split,Articles,Candidates,Positive_1,Negative_0,Pos_Class_Weight
0,train,12,9928,621,9307,14.9871
1,validation,4,1884,131,1753,-


In [3]:
# Section 3: Tokenizer Setup & Dynamic Padding
tokenizer = dl_utils.prepare_tokenizer_and_inputs(
    checkpoint_name=PUBMEDBERT_CHECKPOINT,
    max_length=MAX_LENGTH,
    additional_special_tokens=dl_utils.ENTITY_MARKERS,
    token=False,
)

marker_token_ids = {
    marker: tokenizer.convert_tokens_to_ids(marker)
    for marker in dl_utils.ENTITY_MARKERS
}
assert len(marker_token_ids) == 4
assert all(
    token_id is not None and token_id != tokenizer.unk_token_id
    for token_id in marker_token_ids.values()
), f'Unregistered entity markers: {marker_token_ids}'

class BioCauseDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        encoded = tokenizer(
            list(texts),
            max_length=max_length,
            truncation=True,
        )
        self._input_ids = encoded['input_ids']
        self._attention_mask = encoded['attention_mask']
        self._labels = [int(y) for y in labels]

    def __len__(self):
        return len(self._labels)

    def __getitem__(self, idx):
        if isinstance(idx, str):
            if idx == 'input_ids':
                return self._input_ids
            if idx == 'attention_mask':
                return self._attention_mask
            if idx == 'labels':
                return self._labels
            raise KeyError(idx)
        return {
            'input_ids': self._input_ids[idx],
            'attention_mask': self._attention_mask[idx],
            'labels': self._labels[idx],
        }

train_tokenized = BioCauseDataset(train_df['marked_text'], train_df['label'], tokenizer, MAX_LENGTH)
val_tokenized = BioCauseDataset(val_df['marked_text'], val_df['label'], tokenizer, MAX_LENGTH)

required_marker_ids = list(marker_token_ids.values())
for split_name, dataset, exp_len in [
    ('train', train_tokenized, TARGET_COUNTS['train']['total']),
    ('validation', val_tokenized, TARGET_COUNTS['validation']['total']),
]:
    assert len(dataset) == exp_len, f'{split_name}: length {len(dataset)} != {exp_len}'
    assert max(map(len, dataset['input_ids'])) <= MAX_LENGTH, f'{split_name}: sequence exceeds {MAX_LENGTH}'
    assert set(dataset['labels']) == {0, 1}, f'{split_name}: invalid labels'
    assert all(all(mid in seq for mid in required_marker_ids) for seq in dataset['input_ids']), (
        f'{split_name}: lost entity marker token(s) after tokenization'
    )

In [4]:
# Section 4: Model Instantiation & Embedding Resize
dl_utils.set_seed(SEED)
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        PUBMEDBERT_CHECKPOINT, num_labels=2, local_files_only=True
    )
except Exception:
    model = AutoModelForSequenceClassification.from_pretrained(
        PUBMEDBERT_CHECKPOINT, num_labels=2, token=False
    )
dl_utils.resize_model_embeddings_for_tokenizer(model, tokenizer)
model.to(device)
assert model.config.num_labels == 2
cls_head = model.classifier.out_proj if hasattr(model.classifier, 'out_proj') else model.classifier
assert cls_head.out_features == 2
assert model.get_input_embeddings().weight.shape[0] == len(tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

In [5]:
# Section 5: Correctly Normalized Weighted Loss & Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import inspect
import math
import shutil

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, mean_class_weight=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.mean_class_weight = float(mean_class_weight)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs['labels']
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs = model(**model_inputs)
        logits = outputs.logits
        assert logits.shape[-1] == 2
        assert labels.device == logits.device and labels.dtype in (torch.int32, torch.int64, torch.long)
        weights = self.class_weights.to(device=logits.device, dtype=torch.float32)
        # Apply unweighted per-sample CE first, then apply weights[labels] once (no double weighting).
        per_sample_loss = torch.nn.functional.cross_entropy(
            logits.float(), labels, reduction='none'
        )
        sample_weights = weights[labels]
        # Normalize by the constant training-set mean class weight so random mini-batches
        # preserve the intended ratio w_pos / w_neg = 14.9871 without attenuating positives.
        loss = (per_sample_loss * sample_weights).mean() / self.mean_class_weight
        assert torch.isfinite(loss).all(), f'Non-finite training loss: {loss}'
        return (loss, outputs) if return_outputs else loss

def compute_metrics_with_classes(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    assert logits.shape[-1] == 2
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=[0, 1], zero_division=0
    )
    macro_f1 = float(np.mean(f1))
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'precision_0': float(precision[0]),
        'recall_0': float(recall[0]),
        'f1_0': float(f1[0]),
        'precision_1': float(precision[1]),
        'recall_1': float(recall[1]),
        'f1_1': float(f1[1]),
        'positive_f1': float(f1[1]),
        'macro_f1': macro_f1,
        'f1': macro_f1,
    }

class_weights_tensor = torch.tensor([1.0, pos_weight], dtype=torch.float32)
assert class_weights_tensor.shape == (2,)
assert np.isclose(float(class_weights_tensor[0].item()), 1.0)
assert np.isclose(float(class_weights_tensor[1].item()), 9307 / 621)
assert float(class_weights_tensor[1].item()) > float(class_weights_tensor[0].item())
mean_class_weight = float((n_neg_train * 1.0 + n_pos_train * pos_weight) / len(train_df))
assert np.isclose(mean_class_weight, 2.0 * n_neg_train / len(train_df))

training_args = dl_utils.get_training_arguments(
    output_dir=OUTPUT_DIR, num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, fp16=FP16_ENABLED, eval_strategy=EVAL_STRATEGY,
    save_total_limit=SAVE_TOTAL_LIMIT,
)
training_args.per_device_eval_batch_size = EVAL_BATCH_SIZE
training_args.max_grad_norm = 1.0
# Retain every epoch checkpoint for joint validation (checkpoint, threshold) selection.
training_args.load_best_model_at_end = False
training_args.metric_for_best_model = 'eval_f1_1'
training_args.greater_is_better = True
training_args.save_total_limit = NUM_EPOCHS
total_train_steps = NUM_EPOCHS * max(1, math.ceil(len(train_tokenized) / (BATCH_SIZE * GRAD_ACCUM_STEPS)))
expected_warmup_steps = math.ceil(0.1 * total_train_steps)
training_args.warmup_steps = expected_warmup_steps
assert training_args.get_warmup_steps(total_train_steps) == expected_warmup_steps

trainer_kwargs = {
    'model': model, 'args': training_args, 'train_dataset': train_tokenized,
    'eval_dataset': val_tokenized, 'data_collator': data_collator,
    'compute_metrics': compute_metrics_with_classes,
    'class_weights': class_weights_tensor,
    'mean_class_weight': mean_class_weight,
}
signature = inspect.signature(Trainer.__init__)
if 'processing_class' in signature.parameters:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in signature.parameters:
    trainer_kwargs['tokenizer'] = tokenizer
trainer = WeightedLossTrainer(**trainer_kwargs)

# Pre-training sanity check: verify finite weighted loss and gradient flow on a balanced mini-batch
pos_indices = [i for i, y in enumerate(train_tokenized['labels']) if y == 1][:BATCH_SIZE // 2]
neg_indices = [i for i, y in enumerate(train_tokenized['labels']) if y == 0][:BATCH_SIZE - len(pos_indices)]
check_batch = data_collator([train_tokenized[i] for i in (pos_indices + neg_indices)])
check_batch = {k: v.to(device) for k, v in check_batch.items()}
model.train()
model.zero_grad(set_to_none=True)
check_loss = trainer.compute_loss(model, check_batch)
assert torch.isfinite(check_loss) and float(check_loss.item()) > 0.0
check_loss.backward()
assert cls_head.weight.grad is not None and torch.isfinite(cls_head.weight.grad).all()
assert float(cls_head.weight.grad.abs().sum().item()) > 0.0
emb_grad = model.get_input_embeddings().weight.grad
assert emb_grad is not None and torch.isfinite(emb_grad).all() and float(emb_grad.abs().sum().item()) > 0.0
model.zero_grad(set_to_none=True)
model.eval()
del check_batch, check_loss, emb_grad
dl_utils.set_seed(SEED)

assert training_args.learning_rate == 1e-5
assert training_args.gradient_accumulation_steps == 1
assert training_args.max_grad_norm == 1.0
assert training_args.lr_scheduler_type == 'linear'
assert training_args.load_best_model_at_end is False
assert training_args.metric_for_best_model == 'eval_f1_1'
assert training_args.greater_is_better is True
assert training_args.save_total_limit == NUM_EPOCHS

In [6]:
# Section 6: Training, Joint Epoch-Checkpoint & Threshold Selection on Validation
TRAINING_COMPLETED = False
train_result = trainer.train()
TRAINING_COMPLETED = True

def logits_to_pos_probs(raw_logits):
    logits = raw_logits[0] if isinstance(raw_logits, tuple) else raw_logits
    assert logits.ndim == 2 and logits.shape[1] == 2 and np.isfinite(logits).all()
    logits_f64 = logits.astype(np.float64)
    shifted = logits_f64 - np.max(logits_f64, axis=1, keepdims=True)
    exp_l = np.exp(shifted)
    probs = exp_l / np.sum(exp_l, axis=1, keepdims=True)
    assert np.isfinite(probs).all() and np.allclose(probs.sum(axis=1), 1.0)
    pos_probs = probs[:, 1]
    assert ((pos_probs >= 0.0) & (pos_probs <= 1.0)).all()
    return pos_probs

def build_candidate_thresholds(probs):
    uniform_grid = np.round(np.arange(0.01, 1.00, 0.01), 4)
    quantile_grid = np.round(np.clip(np.quantile(probs, np.linspace(0.01, 0.99, 99)), 0.01, 0.99), 4)
    combined = np.unique(np.concatenate([uniform_grid, quantile_grid]))
    combined = combined[(combined > 0.0) & (combined < 1.0)]
    return [float(t) for t in np.sort(combined)]

def evaluate_threshold(y_true, y_prob, tau):
    pred = y_prob >= tau
    y_pos = y_true == 1
    tp = int(np.sum(pred & y_pos))
    fp = int(np.sum(pred & ~y_pos))
    fn = int(np.sum(~pred & y_pos))
    tn = int(len(y_true) - tp - fp - fn)
    pos_p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    pos_r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pos_f1 = (2.0 * pos_p * pos_r / (pos_p + pos_r)) if (pos_p + pos_r) > 0 else 0.0
    neg_p = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    neg_r = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    neg_f1 = (2.0 * neg_p * neg_r / (neg_p + neg_r)) if (neg_p + neg_r) > 0 else 0.0
    macro_f1 = 0.5 * (neg_f1 + pos_f1)
    return {
        'threshold': float(tau),
        'positive_predictions': int(tp + fp),
        'validation_positive_precision': float(pos_p),
        'validation_positive_recall': float(pos_r),
        'validation_positive_f1': float(pos_f1),
        'validation_macro_f1': float(macro_f1),
    }

epoch_logs = []
latest_train_loss = None
for entry in trainer.state.log_history:
    if 'loss' in entry and 'eval_loss' not in entry:
        latest_train_loss = float(entry['loss'])
    if 'eval_loss' in entry and 'eval_f1_1' in entry:
        epoch_logs.append({
            'epoch': float(entry['epoch']) if entry.get('epoch') is not None else float(len(epoch_logs) + 1),
            'step': int(entry['step']),
            'train_loss': latest_train_loss,
            'validation_loss': float(entry['eval_loss']),
        })
assert len(epoch_logs) == NUM_EPOCHS, f'Expected {NUM_EPOCHS} epoch evaluations, got {len(epoch_logs)}'
assert len({e['step'] for e in epoch_logs}) == NUM_EPOCHS, 'Duplicate global steps in epoch logs'

ckpt_by_step = {}
for p in Path(OUTPUT_DIR).iterdir():
    if p.is_dir() and p.name.startswith('checkpoint-'):
        suffix = p.name[len('checkpoint-'):]
        if not suffix.isdigit():
            raise RuntimeError(f'Ambiguous checkpoint directory name: {p.name}')
        step_num = int(suffix)
        if step_num in ckpt_by_step:
            raise RuntimeError(f'Duplicate checkpoint directory for global step {step_num}')
        ckpt_by_step[step_num] = p

missing_steps = [e['step'] for e in epoch_logs if e['step'] not in ckpt_by_step]
if missing_steps:
    raise RuntimeError(
        f'Missing epoch checkpoint(s) for step(s) {missing_steps}; found {sorted(ckpt_by_step)}'
    )
for e in epoch_logs:
    e['checkpoint_dir'] = str(ckpt_by_step[e['step']])

val_labels_ref = val_df['label'].to_numpy()
assert val_df['candidate_id'].tolist() == AUTHORITATIVE_VAL_CANDIDATE_IDS

joint_epoch_results = []
for e in epoch_logs:
    ckpt_model = AutoModelForSequenceClassification.from_pretrained(e['checkpoint_dir'], local_files_only=True)
    ckpt_model.to(device)
    ckpt_model.eval()
    assert ckpt_model.config.num_labels == 2
    assert ckpt_model.get_input_embeddings().weight.shape[0] == len(tokenizer)
    ckpt_head = ckpt_model.classifier.out_proj if hasattr(ckpt_model.classifier, 'out_proj') else ckpt_model.classifier
    assert ckpt_head.out_features == 2
    assert torch.isfinite(ckpt_head.weight).all() and torch.isfinite(ckpt_head.bias).all()

    trainer.model = ckpt_model
    val_out = trainer.predict(val_tokenized)
    assert np.array_equal(val_out.label_ids, val_labels_ref)
    val_probs_ckpt = logits_to_pos_probs(val_out.predictions)
    candidate_taus = build_candidate_thresholds(val_probs_ckpt)
    ckpt_thresh_records = [evaluate_threshold(val_labels_ref, val_probs_ckpt, tau) for tau in candidate_taus]

    best_ckpt_tau = max(
        ckpt_thresh_records,
        key=lambda r: (
            round(r['validation_positive_f1'], 8),
            round(r['validation_macro_f1'], 8),
            -round(abs(r['threshold'] - 0.50), 6),
            round(r['threshold'], 6),
        ),
    )
    pred_chk = (val_probs_ckpt >= best_ckpt_tau['threshold']).astype(int)
    sk_p, sk_r, sk_f1, _ = precision_recall_fscore_support(val_labels_ref, pred_chk, average='binary', zero_division=0)
    _, _, sk_macro_f1, _ = precision_recall_fscore_support(val_labels_ref, pred_chk, average='macro', zero_division=0)
    assert np.isclose(best_ckpt_tau['validation_positive_precision'], float(sk_p))
    assert np.isclose(best_ckpt_tau['validation_positive_recall'], float(sk_r))
    assert np.isclose(best_ckpt_tau['validation_positive_f1'], float(sk_f1))
    assert np.isclose(best_ckpt_tau['validation_macro_f1'], float(sk_macro_f1))

    joint_epoch_results.append({
        'epoch': e['epoch'],
        'step': e['step'],
        'train_loss': e['train_loss'],
        'validation_loss': e['validation_loss'],
        'best_threshold': float(best_ckpt_tau['threshold']),
        'positive_predictions': int(best_ckpt_tau['positive_predictions']),
        'validation_positive_precision': float(best_ckpt_tau['validation_positive_precision']),
        'validation_positive_recall': float(best_ckpt_tau['validation_positive_recall']),
        'validation_positive_f1': float(best_ckpt_tau['validation_positive_f1']),
        'validation_macro_f1': float(best_ckpt_tau['validation_macro_f1']),
        'checkpoint_dir': e['checkpoint_dir'],
        'val_probs': val_probs_ckpt,
        'threshold_search_records': ckpt_thresh_records,
    })
    del ckpt_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Joint deterministic selection across all saved epoch checkpoints and thresholds:
# 1. Highest validation positive-class F1, 2. Highest validation Macro-F1,
# 3. Lowest validation loss, 4. Deterministic threshold tie-breaker then earlier epoch.
selected_epoch = max(
    joint_epoch_results,
    key=lambda r: (
        round(r['validation_positive_f1'], 8),
        round(r['validation_macro_f1'], 8),
        -round(r['validation_loss'], 8),
        -round(abs(r['best_threshold'] - 0.50), 6),
        round(r['best_threshold'], 6),
        -float(r['epoch']),
    ),
)
FROZEN_BEST_CKPT = str(selected_epoch['checkpoint_dir'])
FROZEN_BEST_TAU = float(selected_epoch['best_threshold'])
FROZEN_SELECTED_EPOCH = float(selected_epoch['epoch'])
FROZEN_SELECTED_STEP = int(selected_epoch['step'])
FROZEN_VAL_POS_F1 = float(selected_epoch['validation_positive_f1'])
FROZEN_VAL_MACRO_F1 = float(selected_epoch['validation_macro_f1'])
FROZEN_VAL_LOSS = float(selected_epoch['validation_loss'])
threshold_search_records = selected_epoch['threshold_search_records']
best_ckpt = FROZEN_BEST_CKPT
best_metric = FROZEN_VAL_POS_F1
best_val_macro_f1_ckpt = FROZEN_VAL_MACRO_F1

if not FROZEN_BEST_CKPT or not Path(FROZEN_BEST_CKPT).exists():
    raise RuntimeError('No validation-selected checkpoint was produced.')
assert np.isfinite(FROZEN_VAL_POS_F1) and np.isfinite(FROZEN_VAL_MACRO_F1) and np.isfinite(FROZEN_VAL_LOSS)

for p in ckpt_by_step.values():
    if str(p) != FROZEN_BEST_CKPT:
        shutil.rmtree(p, ignore_errors=True)

selected_model = AutoModelForSequenceClassification.from_pretrained(FROZEN_BEST_CKPT, local_files_only=True)
selected_model.to(device)
selected_model.eval()
assert selected_model.config.num_labels == 2
assert selected_model.get_input_embeddings().weight.shape[0] == len(tokenizer)
selected_head = selected_model.classifier.out_proj if hasattr(selected_model.classifier, 'out_proj') else selected_model.classifier
assert selected_head.out_features == 2
assert torch.isfinite(selected_head.weight).all() and torch.isfinite(selected_head.bias).all()
trainer.model = selected_model
trainer.state.best_model_checkpoint = FROZEN_BEST_CKPT
trainer.state.best_metric = FROZEN_VAL_POS_F1

display_cols = [
    'epoch', 'step', 'train_loss', 'validation_loss', 'best_threshold',
    'positive_predictions', 'validation_positive_precision',
    'validation_positive_recall', 'validation_positive_f1', 'validation_macro_f1'
]
epoch_metrics_df = pd.DataFrame(joint_epoch_results)[display_cols]
epoch_metrics_df.round(4)

Epoch,Training Loss,Validation Loss,Accuracy,Precision 0,Recall 0,F1 0,Precision 1,Recall 1,F1 1,Positive F1,Macro F1,F1
1,1.366627,1.594957,0.930467,0.930467,1.000000,0.963981,0.000000,0.000000,0.000000,0.000000,0.481991,0.481991
2,1.529718,1.707572,0.930467,0.930467,1.000000,0.963981,0.000000,0.000000,0.000000,0.000000,0.481991,0.481991
3,1.363632,1.573650,0.930467,0.930467,1.000000,0.963981,0.000000,0.000000,0.000000,0.000000,0.481991,0.481991


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,epoch,step,train_loss,validation_loss,best_threshold,positive_predictions,validation_positive_precision,validation_positive_recall,validation_positive_f1,validation_macro_f1
0,1.0,1241,1.3666,1.5950,0.0634,400,0.1300,0.3969,0.1959,0.5320
1,2.0,2482,1.5297,1.7076,0.0343,377,0.2467,0.7099,0.3661,0.6337
2,3.0,3723,1.3636,1.5737,0.0273,377,0.2493,0.7176,0.3701,0.6360


In [7]:
# Section 7: Frozen Validation Diagnostics & Prediction Export
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

assert trainer.state.best_model_checkpoint == FROZEN_BEST_CKPT, 'Selected checkpoint is not active on trainer'
assert trainer.model is selected_model, 'Explicitly loaded checkpoint model is not attached to trainer'
val_predictions = trainer.predict(val_tokenized)
val_dl_prob = logits_to_pos_probs(val_predictions.predictions)
assert np.allclose(val_dl_prob, selected_epoch['val_probs'], atol=1e-6), 'Validation probabilities drifted after reload'

val_labels = val_predictions.label_ids
assert np.array_equal(val_labels, val_df['label'].to_numpy())
assert val_df['candidate_id'].tolist() == AUTHORITATIVE_VAL_CANDIDATE_IDS

val_preds = (val_dl_prob >= FROZEN_BEST_TAU).astype(int)
best_pos_p_val, best_pos_r_val, best_pos_f1_val, _ = precision_recall_fscore_support(
    val_labels, val_preds, average='binary', zero_division=0
)
_, _, best_macro_f1_val, _ = precision_recall_fscore_support(
    val_labels, val_preds, average='macro', zero_division=0
)
assert np.isclose(float(best_pos_f1_val), FROZEN_VAL_POS_F1)
assert np.isclose(float(best_macro_f1_val), FROZEN_VAL_MACRO_F1)

val_cm = confusion_matrix(val_labels, val_preds, labels=[0, 1])
val_tn, val_fp, val_fn, val_tp = [int(x) for x in val_cm.ravel()]

val_preds_export = pd.DataFrame({
    'candidate_id': val_df['candidate_id'].values,
    'article_id': val_df['article_id'].values,
    'section_id': val_df['section_id'].values,
    'arg1_id': val_df['arg1_id'].values,
    'arg2_id': val_df['arg2_id'].values,
    'true_label': val_labels,
    'dl_prob': val_dl_prob,
    'dl_pred': val_preds,
    'split': 'validation',
    'arg1_before_arg2': val_df['arg1_before_arg2'].values,
})
val_pred_path = DATA_DIR / 'dl_biocause_validation_predictions.csv'
dl_utils.export_predictions(val_preds_export, str(val_pred_path), format_type='csv')

val_verify = pd.read_csv(val_pred_path)
assert len(val_verify) == TARGET_COUNTS['validation']['total']
assert val_verify['candidate_id'].tolist() == AUTHORITATIVE_VAL_CANDIDATE_IDS
assert np.isfinite(val_verify['dl_prob']).all() and val_verify['dl_prob'].between(0.0, 1.0).all()

val_threshold_summary_df = pd.DataFrame([{
    'selected_checkpoint': Path(FROZEN_BEST_CKPT).name,
    'selected_epoch': FROZEN_SELECTED_EPOCH,
    'selected_step': FROZEN_SELECTED_STEP,
    'selected_threshold': FROZEN_BEST_TAU,
    'validation_loss': round(FROZEN_VAL_LOSS, 4),
    'val_prob_min': round(float(np.min(val_dl_prob)), 4),
    'val_prob_median': round(float(np.median(val_dl_prob)), 4),
    'val_prob_max': round(float(np.max(val_dl_prob)), 4),
    'positive_predictions': int((val_preds == 1).sum()),
    'validation_positive_precision': round(float(best_pos_p_val), 4),
    'validation_positive_recall': round(float(best_pos_r_val), 4),
    'validation_positive_f1': round(float(best_pos_f1_val), 4),
    'validation_macro_f1': round(float(best_macro_f1_val), 4),
    'tn': val_tn,
    'fp': val_fp,
    'fn': val_fn,
    'tp': val_tp,
    'confusion_matrix': val_cm.tolist(),
}])
val_threshold_summary_df

,selected_checkpoint,selected_epoch,selected_step,selected_threshold,validation_loss,val_prob_min,val_prob_median,val_prob_max,positive_predictions,validation_positive_precision,validation_positive_recall,validation_positive_f1,validation_macro_f1,tn,fp,fn,tp,confusion_matrix
0,checkpoint-3723,3.0,3723,0.0273,1.5737,0.0013,0.0059,0.3182,377,0.2493,0.7176,0.3701,0.636,1470,283,37,94,"[[1470, 283], [37, 94]]"


In [8]:
# Section 8: Held-Out Test Split Evaluation
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

if TRAINING_COMPLETED:
    assert FROZEN_BEST_CKPT and Path(FROZEN_BEST_CKPT).exists(), 'Frozen validation checkpoint missing'
    assert np.isfinite(FROZEN_BEST_TAU) and 0.0 < FROZEN_BEST_TAU < 1.0
    assert np.isfinite(FROZEN_VAL_POS_F1) and np.isfinite(FROZEN_VAL_MACRO_F1) and np.isfinite(FROZEN_VAL_LOSS)
    assert trainer.state.best_model_checkpoint == FROZEN_BEST_CKPT
    assert trainer.model is selected_model

    test_path = resolve_split_path('dl_biocause_test.csv')
    if not test_path.exists():
        raise FileNotFoundError('Missing prepared BioCause test split.')
    AUTHORITATIVE_TEST_CANDIDATE_IDS = pd.read_csv(
        test_path, usecols=['candidate_id']
    )['candidate_id'].tolist()
    test_df = pd.read_csv(test_path)
    dl_utils.validate_dataset_schema(test_df, REQUIRED_COLUMNS, dataset_name='BioCause-test')
    assert 'arg1_before_arg2' in test_df.columns
    assert len(test_df) == TARGET_COUNTS['test']['total']
    assert int((test_df['label'] == 1).sum()) == TARGET_COUNTS['test']['positive']
    assert int((test_df['label'] == 0).sum()) == TARGET_COUNTS['test']['negative']
    assert test_df['candidate_id'].is_unique
    assert test_df['candidate_id'].tolist() == AUTHORITATIVE_TEST_CANDIDATE_IDS
    assert set(test_df['article_id']).isdisjoint(set(train_df['article_id']))
    assert set(test_df['article_id']).isdisjoint(set(val_df['article_id']))
    assert set(test_df['label'].unique()) == {0, 1}
    for marker in dl_utils.ENTITY_MARKERS:
        assert test_df['marked_text'].str.contains(re.escape(marker)).all()
    assert all(
        row.find('[E1]') < row.find('[/E1]') and row[row.find('[E1]') + 4:row.find('[/E1]')].strip() != ''
        and row.find('[E2]') < row.find('[/E2]') and row[row.find('[E2]') + 4:row.find('[/E2]')].strip() != ''
        for row in test_df['marked_text']
    ), 'test: empty or malformed entity span'

    test_tokenized = BioCauseDataset(test_df['marked_text'], test_df['label'], tokenizer, MAX_LENGTH)
    assert len(test_tokenized) == TARGET_COUNTS['test']['total']
    assert max(map(len, test_tokenized['input_ids'])) <= MAX_LENGTH
    assert set(test_tokenized['labels']) == {0, 1}
    assert all(all(mid in seq for mid in required_marker_ids) for seq in test_tokenized['input_ids'])

    test_predictions = trainer.predict(test_tokenized)
    test_labels = test_predictions.label_ids
    assert np.array_equal(test_labels, test_df['label'].to_numpy())
    test_dl_prob = logits_to_pos_probs(test_predictions.predictions)

    test_preds = (test_dl_prob >= FROZEN_BEST_TAU).astype(int)
    test_acc = float(np.mean(test_preds == test_labels))
    test_p, test_r, test_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='macro', zero_division=0
    )
    test_pos_p, test_pos_r, test_pos_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='binary', zero_division=0
    )
    test_roc_auc = float(roc_auc_score(test_labels, test_dl_prob))
    test_pr_auc = float(average_precision_score(test_labels, test_dl_prob))
    cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])
    test_tn, test_fp, test_fn, test_tp = [int(x) for x in cm.ravel()]

    grouped_order_metrics = {}
    for order_val, group_idx in test_df.groupby('arg1_before_arg2').groups.items():
        idx = list(group_idx)
        sub_y = test_labels[idx]
        sub_p = test_preds[idx]
        _, _, sub_macro_f1, _ = precision_recall_fscore_support(sub_y, sub_p, average='macro', zero_division=0)
        gp_p, gp_r, gp_f1, _ = precision_recall_fscore_support(sub_y, sub_p, average='binary', zero_division=0)
        key = 'arg1_before_arg2' if bool(order_val) else 'arg2_before_arg1'
        grouped_order_metrics[key] = {
            'n_examples': int(len(idx)),
            'n_positive': int((sub_y == 1).sum()),
            'accuracy': round(float(np.mean(sub_p == sub_y)), 4),
            'positive_precision': round(float(gp_p), 4),
            'positive_recall': round(float(gp_r), 4),
            'positive_f1': round(float(gp_f1), 4),
            'macro_f1': round(float(sub_macro_f1), 4),
        }

    grouped_article_metrics = {}
    for art_id, group_idx in test_df.groupby('article_id').groups.items():
        idx = list(group_idx)
        sub_y = test_labels[idx]
        sub_p = test_preds[idx]
        _, _, sub_macro_f1, _ = precision_recall_fscore_support(sub_y, sub_p, average='macro', zero_division=0)
        gp_p, gp_r, gp_f1, _ = precision_recall_fscore_support(sub_y, sub_p, average='binary', zero_division=0)
        grouped_article_metrics[str(art_id)] = {
            'n_examples': int(len(idx)),
            'n_positive': int((sub_y == 1).sum()),
            'accuracy': round(float(np.mean(sub_p == sub_y)), 4),
            'positive_precision': round(float(gp_p), 4),
            'positive_recall': round(float(gp_r), 4),
            'positive_f1': round(float(gp_f1), 4),
            'macro_f1': round(float(sub_macro_f1), 4),
        }

    test_summary_df = pd.DataFrame([{
        'split': 'test',
        'checkpoint': Path(FROZEN_BEST_CKPT).name,
        'threshold': FROZEN_BEST_TAU,
        'n_examples': int(len(test_labels)),
        'positive_predictions': int((test_preds == 1).sum()),
        'accuracy': round(test_acc, 4),
        'macro_precision': round(float(test_p), 4),
        'macro_recall': round(float(test_r), 4),
        'macro_f1': round(float(test_f1), 4),
        'positive_precision': round(float(test_pos_p), 4),
        'positive_recall': round(float(test_pos_r), 4),
        'positive_f1': round(float(test_pos_f1), 4),
        'roc_auc': round(test_roc_auc, 4),
        'pr_auc': round(test_pr_auc, 4),
        'tn': test_tn,
        'fp': test_fp,
        'fn': test_fn,
        'tp': test_tp,
        'confusion_matrix': cm.tolist(),
    }])
else:
    test_df = None
    test_tokenized = None
    test_acc = test_p = test_r = test_f1 = None
    test_pos_p = test_pos_r = test_pos_f1 = test_roc_auc = test_pr_auc = None
    cm = np.zeros((2, 2), dtype=int)
    grouped_order_metrics = {}
    grouped_article_metrics = {}
    test_summary_df = pd.DataFrame()

test_summary_df

,split,checkpoint,threshold,n_examples,positive_predictions,accuracy,macro_precision,macro_recall,macro_f1,positive_precision,positive_recall,positive_f1,roc_auc,pr_auc,tn,fp,fn,tp,confusion_matrix
0,test,checkpoint-3723,0.0273,1394,308,0.8221,0.6207,0.8119,0.644,0.2597,0.8,0.3922,0.8809,0.313,1066,228,20,80,"[[1066, 228], [20, 80]]"


In [ ]:
# Section 9: Prediction & Metrics Exports & Model Persistence
EXPORT_DIR = ROOT_DIR / 'data' / 'processed'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if TRAINING_COMPLETED and (test_df is not None):
    # 1. Export test predictions DataFrame (standardized schema matching validation)
    predictions_export = pd.DataFrame({
        'candidate_id': test_df['candidate_id'].values,
        'article_id': test_df['article_id'].values,
        'section_id': test_df['section_id'].values,
        'arg1_id': test_df['arg1_id'].values,
        'arg2_id': test_df['arg2_id'].values,
        'true_label': test_labels,
        'dl_prob': test_dl_prob,
        'dl_pred': test_preds,
        'split': 'test',
        'arg1_before_arg2': test_df['arg1_before_arg2'].values,
    })

    pred_path = EXPORT_DIR / 'dl_biocause_test_predictions.csv'
    assert predictions_export['candidate_id'].tolist() == AUTHORITATIVE_TEST_CANDIDATE_IDS
    dl_utils.export_predictions(predictions_export, str(pred_path), format_type='csv')

    pred_verify = pd.read_csv(pred_path)
    REQUIRED_PRED_COLS = [
        'candidate_id', 'article_id', 'section_id', 'arg1_id', 'arg2_id',
        'true_label', 'dl_prob', 'dl_pred', 'split', 'arg1_before_arg2'
    ]
    assert len(pred_verify) == 1394, f'Expected 1394 rows, got {len(pred_verify)}'
    assert list(pred_verify.columns) == REQUIRED_PRED_COLS, f'Column mismatch: {list(pred_verify.columns)}'
    for col in REQUIRED_PRED_COLS:
        assert pred_verify[col].isna().sum() == 0, f'Found NaN in {col}'
    assert set(pred_verify['true_label'].unique()).issubset({0, 1}), 'Non-binary true_label'
    assert set(pred_verify['dl_pred'].unique()).issubset({0, 1}), 'Non-binary dl_pred'
    assert np.isfinite(pred_verify['dl_prob']).all(), 'Non-finite dl_prob'
    assert pred_verify['dl_prob'].between(0.0, 1.0).all(), 'Probabilities out of [0, 1] range'
    assert pred_verify['candidate_id'].tolist() == AUTHORITATIVE_TEST_CANDIDATE_IDS
    assert (pred_verify['true_label'].values == test_df['label'].values).all()
    assert (pred_verify['split'] == 'test').all()

    # Cross-split schema compatibility assertions
    val_pred_path = EXPORT_DIR / 'dl_biocause_validation_predictions.csv'
    assert val_pred_path.exists(), 'Missing validation prediction export'
    val_check = pd.read_csv(val_pred_path)
    assert val_check['candidate_id'].tolist() == AUTHORITATIVE_VAL_CANDIDATE_IDS
    assert list(val_check.columns) == list(pred_verify.columns), (
        f'Schema mismatch: val {list(val_check.columns)} vs test {list(pred_verify.columns)}'
    )
    for col in REQUIRED_PRED_COLS:
        assert val_check[col].dtype == pred_verify[col].dtype, f'Dtype mismatch in {col}'

    # 2. Export compact metrics JSON
    metrics_export = {
        'task': 'BioCause PubMedBERT Fine-Tuning',
        'model': PUBMEDBERT_CHECKPOINT,
        'training_completed': True,
        'selected_checkpoint': FROZEN_BEST_CKPT,
        'selected_checkpoint_name': Path(FROZEN_BEST_CKPT).name,
        'selected_epoch': float(FROZEN_SELECTED_EPOCH),
        'selected_step': int(FROZEN_SELECTED_STEP),
        'selected_threshold': float(FROZEN_BEST_TAU),
        'decision_threshold': float(FROZEN_BEST_TAU),
        'checkpoint_selection_objective': 'joint_epoch_threshold_max_val_positive_f1_then_macro_f1_then_min_val_loss_then_threshold_then_earlier_epoch',
        'threshold_selection_objective': 'max_val_positive_f1_then_macro_f1_then_closest_to_0.50_then_higher_threshold',
        'split_counts': {
            'train': {'total': len(train_df), 'positive': int((train_df['label'] == 1).sum()), 'negative': int((train_df['label'] == 0).sum())},
            'validation': {'total': len(val_df), 'positive': int((val_df['label'] == 1).sum()), 'negative': int((val_df['label'] == 0).sum())},
            'test': {'total': len(test_df), 'positive': int((test_df['label'] == 1).sum()), 'negative': int((test_df['label'] == 0).sum())},
        },
        'hyperparameters': {
            'num_train_epochs': NUM_EPOCHS,
            'batch_size': BATCH_SIZE,
            'eval_batch_size': EVAL_BATCH_SIZE,
            'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
            'learning_rate': LEARNING_RATE,
            'weight_decay': WEIGHT_DECAY,
            'max_length': MAX_LENGTH,
            'fp16': FP16_ENABLED,
            'seed': SEED,
            'pos_weight': float(pos_weight),
            'mean_class_weight': float(mean_class_weight),
        },
        'best_checkpoint': FROZEN_BEST_CKPT,
        'best_val_positive_f1': float(FROZEN_VAL_POS_F1),
        'best_val_macro_f1': float(FROZEN_VAL_MACRO_F1),
        'best_val_loss': float(FROZEN_VAL_LOSS),
        'validation_diagnostics': {
            'selected_checkpoint': Path(FROZEN_BEST_CKPT).name,
            'selected_epoch': float(FROZEN_SELECTED_EPOCH),
            'selected_step': int(FROZEN_SELECTED_STEP),
            'threshold': float(FROZEN_BEST_TAU),
            'validation_loss': round(float(FROZEN_VAL_LOSS), 4),
            'val_prob_min': round(float(np.min(val_dl_prob)), 4),
            'val_prob_median': round(float(np.median(val_dl_prob)), 4),
            'val_prob_max': round(float(np.max(val_dl_prob)), 4),
            'positive_predictions': int((val_preds == 1).sum()),
            'positive_precision': round(float(best_pos_p_val), 4),
            'positive_recall': round(float(best_pos_r_val), 4),
            'positive_f1': round(float(best_pos_f1_val), 4),
            'macro_f1': round(float(best_macro_f1_val), 4),
            'confusion_matrix': val_cm.tolist(),
        },
        'epoch_metrics': epoch_metrics_df.to_dict(orient='records'),
        'threshold_search_records': threshold_search_records,
        'test_overall': {
            'n_examples': int(len(test_labels)),
            'positive_predictions': int((test_preds == 1).sum()),
            'accuracy': round(float(test_acc), 4),
            'macro_precision': round(float(test_p), 4),
            'macro_recall': round(float(test_r), 4),
            'macro_f1': round(float(test_f1), 4),
            'positive_precision': round(float(test_pos_p), 4),
            'positive_recall': round(float(test_pos_r), 4),
            'positive_f1': round(float(test_pos_f1), 4),
            'roc_auc': round(float(test_roc_auc), 4),
            'pr_auc': round(float(test_pr_auc), 4),
        },
        'confusion_matrix': cm.tolist(),
        'test_by_argument_order': grouped_order_metrics,
        'test_by_article': grouped_article_metrics,
    }

    metrics_path = EXPORT_DIR / 'dl_biocause_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics_export, f, indent=2, default=lambda x: x.item() if hasattr(x, 'item') else str(x))
    assert metrics_path.exists(), f'Missing metrics export: {metrics_path.name}'

    # 3. Memory-safe CPU persistence (SafeTensors + tokenizer) & reload verification
    import gc
    BEST_MODEL_DIR = ROOT_DIR / 'DL' / 'models' / 'biocause_pubmedbert_best'
    HYBRID_PUBMEDBERT_DIR = ROOT_DIR / 'DL' / 'models' / 'hybrid_biocause' / 'pubmedbert'
    if 'trainer' in globals() and trainer is not None:
        trainer.optimizer = None
        trainer.lr_scheduler = None
    if 'selected_model' in globals() and selected_model is not None:
        selected_model.to('cpu')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    val_sanity_texts = val_df['marked_text'].iloc[:4].astype(str).tolist()
    try:
        from hybrid_inference import persist_and_verify_pubmedbert
        artifact_inventory = persist_and_verify_pubmedbert(
            source_checkpoint_dir=FROZEN_BEST_CKPT,
            output_dirs=[BEST_MODEL_DIR, HYBRID_PUBMEDBERT_DIR],
            sample_texts=val_sanity_texts,
            max_length=MAX_LENGTH,
        )
    except ImportError:
        cpu_model = AutoModelForSequenceClassification.from_pretrained(FROZEN_BEST_CKPT, local_files_only=True).to('cpu')
        cpu_model.eval()
        artifact_inventory = []
        for out_dir in (BEST_MODEL_DIR, HYBRID_PUBMEDBERT_DIR):
            out_dir.mkdir(parents=True, exist_ok=True)
            try:
                cpu_model.save_pretrained(str(out_dir), safe_serialization=True)
            except Exception:
                cpu_model.save_pretrained(str(out_dir), safe_serialization=False)
            tokenizer.save_pretrained(str(out_dir))
            assert (out_dir / 'config.json').is_file()
            assert (out_dir / 'model.safetensors').is_file() or (out_dir / 'pytorch_model.bin').is_file()
            re_tok = AutoTokenizer.from_pretrained(str(out_dir), local_files_only=True)
            re_mod = AutoModelForSequenceClassification.from_pretrained(str(out_dir), local_files_only=True)
            re_mod.eval()
            assert re_mod.config.num_labels == 2
            assert re_mod.get_input_embeddings().weight.shape[0] == len(re_tok)
            with torch.no_grad():
                enc = re_tok(val_sanity_texts, max_length=MAX_LENGTH, truncation=True, padding=True, return_tensors='pt')
                probs_check = torch.softmax(re_mod(**enc).logits.float(), dim=-1)[:, 1].cpu().numpy()
            assert probs_check.shape == (4,) and np.isfinite(probs_check).all()
            for fp in sorted(out_dir.rglob('*')):
                if fp.is_file():
                    artifact_inventory.append({'directory': out_dir.name, 'file': str(fp.relative_to(out_dir)), 'size_bytes': int(fp.stat().st_size)})
            del re_mod, re_tok
        del cpu_model
        gc.collect()
    assert (BEST_MODEL_DIR / 'config.json').is_file(), 'Saved model config.json missing'
    pd.DataFrame(artifact_inventory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

: 

## 4.10 Standalone Checkpoint Recovery & SafeTensors Persistence Verification (No Retraining)

If the Colab runtime ran out of memory during model shard writing in Section 9, run the recovery cell below **without retraining**.
It reads the frozen checkpoint identifier from `data/processed/dl_biocause_metrics.json`, loads that exact epoch checkpoint on CPU, saves weights in SafeTensors format along with the special-marker tokenizer into `DL/models/biocause_pubmedbert_best/` and `DL/models/hybrid_biocause/pubmedbert/`, reloads from disk, verifies vocabulary size and classifier head dimensions, and runs a 4-sample deterministic validation sanity check (no test labels accessed).

In [ ]:
# Section 10: Standalone Checkpoint Recovery & SafeTensors Persistence Verification
import gc
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if 'ROOT_DIR' not in globals():
    curr = Path(os.getcwd()).resolve()
    search_roots = [
        Path('/content/causal-nlp-reasoning'),
        Path('/workspace/causal-nlp-reasoning'),
        Path('/kaggle/working/causal-nlp-reasoning'),
        curr,
        Path('/content'),
        Path('/workspace'),
        Path('/kaggle/working'),
    ]
    ROOT_DIR = None
    for base in search_roots:
        if not base.exists():
            continue
        cands = [base, *base.parents]
        if base in {Path('/content'), Path('/workspace'), Path('/kaggle/working')}:
            cands.extend(p.parent.parent for p in base.rglob('DL/dl_utils.py'))
        for cand in cands:
            if (cand / 'AML' / 'causal_extractors.py').is_file() and (cand / 'DL' / 'dl_utils.py').is_file():
                ROOT_DIR = cand.resolve()
                break
        if ROOT_DIR is not None:
            break
    if ROOT_DIR is None:
        raise FileNotFoundError('Repository root not found.')

for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils

dl_utils.set_seed(42)
DATA_DIR = ROOT_DIR / 'data' / 'processed'
METRICS_PATH = DATA_DIR / 'dl_biocause_metrics.json'
if not METRICS_PATH.is_file():
    raise FileNotFoundError(f'Missing metrics metadata required for recovery: {METRICS_PATH}')

metrics_meta = json.loads(METRICS_PATH.read_text())
frozen_ckpt_name = str(metrics_meta['selected_checkpoint_name'])
recorded_ckpt_path = Path(str(metrics_meta['selected_checkpoint']))
best_model_dir = ROOT_DIR / 'DL' / 'models' / 'biocause_pubmedbert_best'
hybrid_pubmed_dir = ROOT_DIR / 'DL' / 'models' / 'hybrid_biocause' / 'pubmedbert'

candidate_sources = [
    recorded_ckpt_path,
    ROOT_DIR / 'DL' / 'checkpoints' / 'biocause_pubmedbert' / frozen_ckpt_name,
]
if (best_model_dir / 'config.json').is_file() and (
    (best_model_dir / 'model.safetensors').is_file() or (best_model_dir / 'pytorch_model.bin').is_file()
):
    candidate_sources.append(best_model_dir)

frozen_source_dir = next((p for p in candidate_sources if p.is_dir() and (p / 'config.json').is_file()), None)
if frozen_source_dir is None:
    raise FileNotFoundError(
        f'Frozen validation-selected checkpoint {frozen_ckpt_name!r} was not found at '
        f'{[str(p) for p in candidate_sources]}. Re-run training in Colab without substituting base weights.'
    )

val_manifest = pd.read_csv(DATA_DIR / 'dl_biocause_val.csv')
assert len(val_manifest) == dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['validation']['total']
sanity_texts = val_manifest['marked_text'].iloc[:4].astype(str).tolist()
assert all(all(m in txt for m in dl_utils.ENTITY_MARKERS) for txt in sanity_texts)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

tok = AutoTokenizer.from_pretrained(str(frozen_source_dir), local_files_only=True)
for marker in dl_utils.ENTITY_MARKERS:
    tid = tok.convert_tokens_to_ids(marker)
    assert tid is not None and tid != tok.unk_token_id, f'Missing marker {marker} in checkpoint tokenizer'

mdl = AutoModelForSequenceClassification.from_pretrained(str(frozen_source_dir), local_files_only=True)
mdl.to('cpu')
mdl.eval()

verification_rows = []
for target_dir in (best_model_dir, hybrid_pubmed_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    if frozen_source_dir.resolve() != target_dir.resolve():
        try:
            mdl.save_pretrained(str(target_dir), safe_serialization=True)
        except Exception:
            mdl.save_pretrained(str(target_dir), safe_serialization=False)
        tok.save_pretrained(str(target_dir))

    assert (target_dir / 'config.json').is_file(), f'Missing config.json in {target_dir}'
    weight_file = 'model.safetensors' if (target_dir / 'model.safetensors').is_file() else 'pytorch_model.bin'
    assert (target_dir / weight_file).is_file(), f'Missing weight file in {target_dir}'
    assert (target_dir / 'tokenizer_config.json').is_file(), f'Missing tokenizer_config.json in {target_dir}'

    reloaded_tok = AutoTokenizer.from_pretrained(str(target_dir), local_files_only=True)
    reloaded_mdl = AutoModelForSequenceClassification.from_pretrained(str(target_dir), local_files_only=True)
    reloaded_mdl.to('cpu')
    reloaded_mdl.eval()

    assert reloaded_mdl.config.num_labels == 2
    assert reloaded_mdl.get_input_embeddings().weight.shape[0] == len(reloaded_tok)
    head = reloaded_mdl.classifier.out_proj if hasattr(reloaded_mdl.classifier, 'out_proj') else reloaded_mdl.classifier
    assert head.out_features == 2
    assert torch.isfinite(head.weight).all() and torch.isfinite(head.bias).all()

    with torch.no_grad():
        batch = reloaded_tok(sanity_texts, max_length=384, truncation=True, padding=True, return_tensors='pt')
        logits = reloaded_mdl(**batch).logits
        probs_a = torch.softmax(logits.float(), dim=-1)[:, 1].numpy()
        probs_b = torch.softmax(reloaded_mdl(**batch).logits.float(), dim=-1)[:, 1].numpy()
    assert probs_a.shape == (4,) and np.isfinite(probs_a).all()
    assert ((probs_a >= 0.0) & (probs_a <= 1.0)).all()
    assert np.allclose(probs_a, probs_b, atol=1e-7), 'Non-deterministic eval sanity check'

    for fp in sorted(target_dir.iterdir()):
        if fp.is_file():
            verification_rows.append({
                'Target': target_dir.name,
                'File': fp.name,
                'Size_Bytes': int(fp.stat().st_size),
                'Size_MB': round(float(fp.stat().st_size) / (1024.0 * 1024.0), 4),
                'Vocab_Size': len(reloaded_tok),
                'Num_Labels': int(reloaded_mdl.config.num_labels),
                'Sanity_Prob_Range': f'{probs_a.min():.4f}..{probs_a.max():.4f}',
                'Status': 'VERIFIED',
            })
    del reloaded_mdl, reloaded_tok

del mdl, tok
gc.collect()
pd.DataFrame(verification_rows)

## 5. Expected Outputs & Evaluation Criteria

- Best fine-tuned PubMedBERT checkpoint.
- `data/processed/dl_biocause_validation_predictions.csv`: 1,884 validation candidates with raw probabilities and decisions.
- `data/processed/dl_biocause_test_predictions.csv`: 1,394 test candidates with raw probabilities and decisions.
- **Evaluation Criteria**: Candidate-level Precision, Recall, and F1 on the 4 held-out test publications (100 gold relations, 1,394 candidates).
